# PRISM: QLoRA fine-tuning of Phi-3.5-mini (Kaggle, GPU T4 x1 or P100)

1. Build the silver data locally: `python training/build_dataset.py --pool 800`.
2. Upload `training/data/sft.jsonl` and `sft_report.json` as a Kaggle dataset named **prism-sft**, and add it to this notebook.
3. Settings → Accelerator: GPU; Internet: on. Run all cells.
4. Download `prism-legal-q4_k_m.gguf` and `train_summary.json` from the Output tab, then locally:
   `python training/export_ollama.py register --gguf prism-legal-q4_k_m.gguf`.

In [ ]:
!git clone --depth 1 https://github.com/febinrenu/PRISM.git
!pip install -q -r PRISM/training/requirements-train.txt

In [ ]:
import glob, os, shutil
src = glob.glob('/kaggle/input/*/sft.jsonl')
assert src, 'Add the prism-sft dataset (sft.jsonl) to this notebook'
os.makedirs('PRISM/training/data', exist_ok=True)
shutil.copy(src[0], 'PRISM/training/data/sft.jsonl')
!wc -l PRISM/training/data/sft.jsonl

Smoke run: a few steps on 16 examples; fails unless every LoRA module type (qkv, o, gate_up, down) is updated.

In [ ]:
!cd PRISM && python training/finetune.py --smoke

Full run (about 1–2 hours on a T4 for ~500 examples, 2 epochs).

In [ ]:
!cd PRISM && python training/finetune.py --epochs 2

In [ ]:
!cat PRISM/training/prism-phi35-lora/train_summary.json | head -40

Merge, convert to GGUF and quantise to Q4_K_M with llama.cpp.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!cd PRISM && python training/export_ollama.py merge --llama-cpp ../llama.cpp

In [ ]:
for f in ['PRISM/training/prism-legal-q4_k_m.gguf', 'PRISM/training/prism-phi35-lora/train_summary.json']:
    shutil.copy(f, '/kaggle/working/')
!ls -la /kaggle/working